# Semana 10 · Sesión 1: Del notebook largo al proyecto

**Módulo 2**

## Objetivos de la sesión

1. Reconocer los síntomas de un notebook largo: **estado oculto**, celdas que
   dependen del orden en que se corrieron y código copiado y pegado.
2. Mover funciones a un **módulo `.py`** e importarlas desde el notebook.
3. Organizar un proyecto en README, módulo y notebooks numerados, que cada uno
   corra solo desde un kernel limpio.

## Retomamos

Desde la semana 4 cada notebook fue una sola historia de hora y media. Un
proyecto de física de verdad, como el **proyecto intermedio** que arranca esta
semana, no cabe así: crece a cientos de celdas, se reescribe varias veces y lo
lee alguien más (tu pareja, quien lo revisa, tú dentro de un mes).

Hoy no hay física nueva. Usamos un sistema que ya conoces, el **oscilador
amortiguado**, $m\ddot x + b\dot x + kx = 0$, que se resuelve con el `dsolve`
de la semana 5, y nos concentramos en *dónde* vive el código.

In [ ]:
import sympy as sp

sp.init_printing()

## Síntomas de un notebook largo

Jupyter ejecuta las celdas **en el orden en que tú las corres**, no en el orden
en que aparecen en la pantalla. Lo que una celda deja en memoria (el *estado*
del kernel) lo ve cualquier celda que se corra después, esté arriba o abajo.
Mientras el notebook es corto, eso no se nota.

Empecemos con la frecuencia natural $\omega_0 = \sqrt{k/m}$ de un resorte
concreto. Usamos enteros de SymPy para que la raíz quede exacta.

In [ ]:
k = sp.Integer(4)   # constante del resorte, en N/m
m = sp.Integer(1)   # masa, en kg

omega_0 = sp.sqrt(k / m)
display(sp.Eq(sp.Symbol("omega_0"), omega_0))

Cuarenta celdas más abajo, alguien necesitó la constante del resorte como
símbolo para derivar una fórmula general, y reutilizó el nombre `k`:

In [ ]:
k = sp.Symbol("k", positive=True)   # positiva: un resorte siempre restaura

Ahora vuelve a correr **la misma** celda de $\omega_0$. Aquí la copiamos para
que el notebook ejecute en orden; en clase basta con subir y darle
`Shift+Enter` a la de arriba.

In [ ]:
omega_0 = sp.sqrt(k / m)   # idéntica a la de arriba
display(sp.Eq(sp.Symbol("omega_0"), omega_0))

La misma celda dio $2$ y luego $\sqrt{k}$. El resultado depende de **qué se
corrió antes**, y eso no queda escrito en ningún lado. A eso se le llama
**estado oculto**.

Su primo es la celda borrada: definiste `periodo` en una celda, la usaste más
abajo y luego borraste la celda que lo definía. Mientras el kernel siga vivo,
todo funciona. En cuanto alguien abre el notebook desde cero:

In [ ]:
try:
    display(periodo)   # se definía en una celda que ya no existe
except NameError as error:
    print("NameError:", error)

La prueba que atrapa los dos síntomas es la misma: **Kernel → Restart Kernel
and Run All Cells**. Si el notebook llega al final sin errores, lo que ves es
lo que hay. Hazla antes de cada commit.

El tercer síntoma es más silencioso: **copiar y pegar**. Queremos resolver el
oscilador en sus tres regímenes, que dependen del signo de $b^2 - 4mk$. Con
$m = 1$, $k = 4$, $x(0) = 1$ y $\dot x(0) = 0$, para eso solo hay que cambiar $b$:

In [ ]:
t = sp.Symbol("t", nonnegative=True)   # el tiempo no es negativo
x = sp.Function("x", real=True)        # la posición es real

In [ ]:
m, b, k = 1, 1, 4   # subamortiguado: b² < 4mk
ecuacion = sp.Eq(m * x(t).diff(t, 2) + b * x(t).diff(t) + k * x(t), 0)
solucion = sp.dsolve(ecuacion, x(t), ics={x(0): 1, x(t).diff(t).subs(t, 0): 0})
display(sp.simplify(solucion))

In [ ]:
m, b, k = 1, 4, 4   # crítico: b² = 4mk
ecuacion = sp.Eq(m * x(t).diff(t, 2) + b * x(t).diff(t) + k * x(t), 0)
solucion = sp.dsolve(ecuacion, x(t), ics={x(0): 1, x(t).diff(t).subs(t, 0): 0})
display(sp.simplify(solucion))

In [ ]:
m, b, k = 1, 5, 4   # sobreamortiguado: b² > 4mk
ecuacion = sp.Eq(m * x(t).diff(t, 2) + b * x(t).diff(t) + k * x(t), 0)
solucion = sp.dsolve(ecuacion, x(t), ics={x(0): 1, x(t).diff(t).subs(t, 0): 0})
display(sp.simplify(solucion))

Funciona, pero las tres celdas son la misma con un número distinto. Si mañana
quieres otra condición inicial, hay que editar tres lugares y acordarse de los
tres. Y fíjate que cada celda reescribe `m`, `b` y `k`: más estado oculto para
la celda que venga después.

## TODO en clase 1

Junta las tres celdas en una función.

1. Escribe `resolver_oscilador(m, b, k, x0, v0)` que devuelva la expresión
   $x(t)$, ya simplificada (el lado derecho de la solución: `.rhs`), con
   $x(0) = x_0$ y $\dot x(0) = v_0$. Dale un docstring en español que diga qué
   resuelve.
2. Usa la función en un ciclo sobre `[1, 4, 5]` para obtener las tres
   soluciones y guárdalas en la lista `soluciones`.
3. Muestra cada una como ecuación, con `display(sp.Eq(x(t), ...))`.

*Pista:* las variables que se crean dentro de la función no se escapan al
notebook. Una función también es una forma de **no** dejar estado oculto.

In [ ]:
# TODO en clase: una función en lugar de tres celdas
def resolver_oscilador(m, b, k, x0, v0):
    """Escribe aquí qué resuelve la función."""
    ...


soluciones = ...

## Un módulo `.py`

La función ya existe, pero vive dentro de este notebook. Si mañana el
notebook de las gráficas la necesita, ¿la copias? Así empieza otra vez el
problema.

La salida es un **módulo**: un archivo `.py` con funciones, sin celdas. Junto a
este notebook está [`oscilador.py`](oscilador.py). Ábrelo en JupyterLab con
doble clic y léelo: tiene `ecuacion_de_movimiento`, `regimen` y `resolver`,
cada una con su docstring, y los símbolos `t` y `x` definidos una sola vez.

Python encuentra el módulo porque está en la **misma carpeta** que el
notebook. Antes de importarlo activamos `autoreload`: sin él, Python lee el
archivo una sola vez por kernel y no se entera de que lo editaste.

In [ ]:
%load_ext autoreload
%autoreload 2

import oscilador

En un proyecto, `import oscilador` va en la celda de imports de arriba, junto
a `sympy`. Aquí lo dejamos en su sección para verlo aparecer.

Un módulo es un **espacio de nombres**: todo lo que define se usa con el
prefijo `oscilador.`, y no se mezcla con las variables del notebook. Qué
contiene:

In [ ]:
print([nombre for nombre in dir(oscilador) if not nombre.startswith("_")])

Ahí aparece también `sp`: el módulo importó SymPy para sí mismo. Los
docstrings se leen sin abrir el archivo:

In [ ]:
help(oscilador.resolver)

Con símbolos positivos (masa, fricción y resorte lo son), el módulo da la
ecuación general. Usamos `oscilador.t` y `oscilador.x`, los del módulo, para no
depender de que los del notebook coincidan.

In [ ]:
m, b, k = sp.symbols("m b k", positive=True)   # las tres constantes son positivas
display(oscilador.ecuacion_de_movimiento(m, b, k))

In [ ]:
for friccion in [1, 4, 5]:
    print(oscilador.regimen(1, friccion, 4))
    display(sp.Eq(oscilador.x(oscilador.t), oscilador.resolver(1, friccion, 4, 1, 0)))

Las tres celdas copiadas se volvieron tres líneas, y cualquier notebook que
haga `import oscilador` obtiene lo mismo.

## TODO en clase 2

Ahora **editas el módulo**, no el notebook.

1. En `oscilador.py`, agrega una función `energia_mecanica(m, k, posicion)` con
   su docstring, que devuelva $E = \tfrac12 m\dot x^2 + \tfrac12 k x^2$ para una
   `posicion` que sea una expresión en `t`. Guarda el archivo (`Ctrl+S`).
2. De vuelta aquí, con la `posicion` del caso subamortiguado de abajo, calcula
   `energia` con `oscilador.energia_mecanica`.
3. Calcula `energia_inicial`, que es $E(0)$. Debe dar $2$: toda la energía está
   en el resorte, $\tfrac12 \cdot 4 \cdot 1^2$.
4. La fricción disipa energía a razón de $\dot E = -b\dot x^2$. Calcula
   `perdida` $= \dot E + b\dot x^2$, con $b = 1$, y simplifícala: debe dar $0$.

*Pista:* gracias a `autoreload` no hace falta reiniciar el kernel; basta con
guardar el `.py` y volver a correr la celda.

In [ ]:
# TODO en clase: la energía mecánica, calculada en el módulo
posicion = oscilador.resolver(1, 1, 4, 1, 0)   # m = 1, b = 1, k = 4

energia = ...
energia_inicial = ...
perdida = ...

## Un proyecto en varios notebooks

Con un módulo a la mano, el proyecto ya no tiene que ser un notebook enorme.
Esta es la forma que te pedimos para el proyecto intermedio:

![Estructura de un proyecto: README, requirements, un módulo y tres notebooks numerados que lo importan](img/estructura-proyecto.svg)

- **`modelo.py`**: las definiciones (símbolos con sus suposiciones, funciones,
  parámetros numéricos). Nada de `display`: el módulo calcula y los notebooks
  muestran.
- **Notebooks numerados** en el orden en que se leen: el planteamiento, la
  solución, las gráficas. Cada uno importa el módulo y **corre solo** desde un
  kernel limpio; ninguno depende de lo que otro dejó en memoria.
- **Una sola fuente de verdad**: si $k = 4$, se escribe una vez, en el módulo.
  Si aparece en dos notebooks, algún día van a diferir.
- **`README.md`**: qué sistema es, qué pregunta responde y cómo se corre.
- **`requirements.txt`**: las versiones con las que funciona, como el del curso.

¿Cómo saber qué va al módulo y qué se queda en el notebook? Una regla útil:
si lo vas a usar en **más de un** notebook, o si es una función que podrías
explicar con un docstring, va al módulo. La narración, las celdas que muestran
resultados y las gráficas se quedan en el notebook.

Todo esto también cambia los commits. Un cambio en `modelo.py` se lee en el
diff como código normal, línea por línea, sin el JSON del notebook alrededor.
En la sesión 2 eso importa, porque alguien más va a leer tu diff.

## TODO en clase 3

Piensa en tu proyecto intermedio (las instrucciones completas se ven en la
sesión 2). Contesta en la celda de abajo:

1. Uno o dos sistemas físicos candidatos: un péndulo con algo extra, un
   circuito, una órbita, un sistema cuántico de dos niveles, lo que te
   interese de las semanas 4 a 9.
2. Para el que más te guste, el árbol de archivos de tu proyecto, como el
   diagrama de arriba.
3. Tres funciones que irían en tu `modelo.py`, con el nombre y una línea de
   docstring cada una.

*Tu respuesta aquí.*

## Resumen

Un notebook largo se rompe de tres maneras silenciosas: **estado oculto**
(el resultado depende del orden de ejecución), **celdas borradas** (algo se
definía en una celda que ya no existe) y **código copiado** (el mismo cálculo
en tres lugares). *Restart & Run All* detecta los dos primeros; las
**funciones** curan el tercero.

Un **módulo `.py`** junto al notebook se importa con `import`; con
`%autoreload 2` los cambios al archivo se ven sin reiniciar. Un proyecto
organizado tiene README, `requirements.txt`, un módulo y notebooks numerados
que lo importan y corren solos.

**Siguiente sesión:** el código ya no es solo tuyo. Cómo revisar el Pull
Request de un compañero, cómo responder a una revisión y cómo arrancar el
repositorio de tu proyecto intermedio.